# EDA 01: calidad de datos

Este notebook describe el dataset **original**. No elimina duplicados, no imputa valores y no sobrescribe `data/raw/heart.csv`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data' / 'raw').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data_quality import quality_summary, frequency_table, suspicious_values

TABLES = ROOT / 'outputs' / 'tables'
TABLES.mkdir(parents=True, exist_ok=True)
DATA_PATH = ROOT / 'data' / 'raw' / 'heart.csv'
df = pd.read_csv(DATA_PATH)
print(f'Ruta: {DATA_PATH}')
print(f'Forma: {df.shape}')

Ruta: C:\Users\cesar\OneDrive\Desktop\DuocUC\3erYear\GestionDeProyectoDeDatos\eva1\data\raw\heart.csv
Forma: (1025, 14)


## Vista y estructura

Se revisan las primeras filas, dimensiones, columnas, tipos y estadísticos descriptivos de la fuente original.

In [2]:
display(df.head())
print('shape:', df.shape)
print('columns:', df.columns.tolist())
df.info()
display(df.describe().T)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


shape: (1025, 14)
columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
<class 'pandas.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   int64  
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB


,count,mean,std,min,25%,50%,75%,max
age,1025.0,54.434146,9.072290,29.0,48.0,56.0,61.0,77.0
sex,1025.0,0.695610,0.460373,0.0,0.0,1.0,1.0,1.0
cp,1025.0,0.942439,1.029641,0.0,0.0,1.0,2.0,3.0
trestbps,1025.0,131.611707,17.516718,94.0,120.0,130.0,140.0,200.0
chol,1025.0,246.000000,51.592510,126.0,211.0,240.0,275.0,564.0
fbs,1025.0,0.149268,0.356527,0.0,0.0,0.0,0.0,1.0
restecg,1025.0,0.529756,0.527878,0.0,0.0,1.0,1.0,2.0
thalach,1025.0,149.114146,23.005724,71.0,132.0,152.0,166.0,202.0
exang,1025.0,0.336585,0.472772,0.0,0.0,0.0,1.0,1.0
oldpeak,1025.0,1.071512,1.175053,0.0,0.0,0.8,1.8,6.2


## Completitud, duplicados y resumen de variables

Estas métricas son descriptivas. Los duplicados se cuantifican, pero no se eliminan.

In [3]:
nulls = pd.DataFrame({'nulos': df.isna().sum(), 'porcentaje_nulos': (df.isna().mean() * 100).round(2)})
display(nulls)
print('Filas duplicadas exactas:', int(df.duplicated().sum()))
summary = quality_summary(df)
display(summary)
summary.to_csv(TABLES / 'calidad_datos.csv', index=False)
summary.to_csv(TABLES / 'resumen_variables.csv', index=False)
nulls.reset_index(names='variable').to_csv(TABLES / 'nulos_por_columna.csv', index=False)

,nulos,porcentaje_nulos
age,0,0.0
sex,0,0.0
cp,0,0.0
trestbps,0,0.0
chol,0,0.0
fbs,0,0.0
restecg,0,0.0
thalach,0,0.0
exang,0,0.0
oldpeak,0,0.0


Filas duplicadas exactas: 723


,variable,tipo_dato,nulos,porcentaje_nulos,valores_unicos,minimo,maximo
0,age,int64,0,0.0,41,29.0,77.0
1,sex,int64,0,0.0,2,0.0,1.0
2,cp,int64,0,0.0,4,0.0,3.0
3,trestbps,int64,0,0.0,49,94.0,200.0
4,chol,int64,0,0.0,152,126.0,564.0
5,fbs,int64,0,0.0,2,0.0,1.0
6,restecg,int64,0,0.0,3,0.0,2.0
7,thalach,int64,0,0.0,91,71.0,202.0
8,exang,int64,0,0.0,2,0.0,1.0
9,oldpeak,float64,0,0.0,40,0.0,6.2


## Frecuencias de variables categóricas o discretas

Se exportan frecuencias para variables con diez o menos valores distintos; esto permite revisar concentración y grupos pequeños sin modificar la fuente.

In [4]:
discrete = [c for c in df.columns if df[c].nunique() <= 10]
for column in discrete:
    table = frequency_table(df, column)
    table.to_csv(TABLES / f'frecuencias_{column}.csv', index=False)
    print(f'\n{column}')
    display(table)
print('Variables discretas revisadas:', discrete)


sex


,valor,conteo,porcentaje
0,0,312,30.44
1,1,713,69.56



cp


,valor,conteo,porcentaje
0,0,497,48.49
1,1,167,16.29
2,2,284,27.71
3,3,77,7.51



fbs


...


thal


,valor,conteo,porcentaje
0,0,7,0.68
1,1,64,6.24
2,2,544,53.07
3,3,410,40.00



target


,valor,conteo,porcentaje
0,0,499,48.68
1,1,526,51.32


Variables discretas revisadas: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal', 'target']


## Rangos y revisión de valores sospechosos

La revisión aplica rangos amplios de plausibilidad y marca únicamente valores que requieren revisión. No corrige ni excluye registros.

In [5]:
ranges = df.select_dtypes(include=np.number).agg(['min', 'max']).T
display(ranges)
review = suspicious_values(df)
display(review)
ranges.reset_index(names='variable').to_csv(TABLES / 'rangos_numericos.csv', index=False)
review.to_csv(TABLES / 'valores_sospechosos.csv', index=False)
print('Archivos de calidad exportados en:', TABLES)

,min,max
age,29.0,77.0
sex,0.0,1.0
cp,0.0,3.0
trestbps,94.0,200.0
chol,126.0,564.0
fbs,0.0,1.0
restecg,0.0,2.0
thalach,71.0,202.0
exang,0.0,1.0
oldpeak,0.0,6.2


,variable,rango_revisado,valores_fuera_rango,requiere_revision
0,age,0 a 120,0,No
1,trestbps,0 a 300,0,No
2,chol,0 a 1000,0,No
3,thalach,0 a 250,0,No
4,oldpeak,0 a 20,0,No


Archivos de calidad exportados en:

 C:\Users\cesar\OneDrive\Desktop\DuocUC\3erYear\GestionDeProyectoDeDatos\eva1\outputs\tables
